Code to mix pioreactor samples, dilute them across two plates and then transfer samples to PCR plates by way of a master mix plate.

Deck setup:
Rail 1 - PCR plate carrier
Rail 7
Rail 13
Rail 19
Rail 25 - Tip carrier [1000, 50, 10, unused, 50]

In [1]:
###Basic structure of code
#Make PCR master plate (may do this by hand, want to reduce ptentail contimation)
# Dilute plate
# dilute plate into ditution plate
# Dump tips
# Diltuion plate to PCR master plate
#PCR master plate to sample plates
#Repeat for the whole sample plate


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, hamilton_mfx_carrier_L5_base, TIP_CAR_480_A00, PLT_CAR_L5MD_A00
from pylabrobot.resources.hamilton.mfx_modules import hamilton_mfx_plateholder_DWP_metal_tapped
#from pylabrobot.resources.opentrons.tube_racks import (
#    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
#)
# from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL_filter,     # 1000 µL filtered 
    hamilton_96_tiprack_10uL_filter #Tip Rack with 96 10ul Low Volume Tip with filter
)
from pylabrobot.liquid_handling.standard import Mix #Need this mix method to mix later

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

2026-03-05 12:42:12,136 - pylabrobot.io.usb - INFO - Finding USB device...
2026-03-05 12:42:12,148 - pylabrobot.io.usb - INFO - Found USB device.
2026-03-05 12:42:12,151 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-03-05 12:42:15,314 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

def VWR_96_wellplate_100_Vb_on_starCarrier_182070(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
It is currently on a STARLet carrier with catalog number 182070, and the plate is modeled on the carrier.
  """
  
  return Plate(
    name=name,
    size_x=127.55,
    size_y=85.0,
    size_z=18.6,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb_on_starCarrier_182070.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11,  # measured
      dy=10,  # measured
      dz=2.15, # measured
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=18, # measured well depth
      material_z_thickness=1.0,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

In [5]:
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

# Tip racks on rails=25, module slots 0,1,2,4
tiprack_1000 = hamilton_96_tiprack_1000uL_filter("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_pcr = hamilton_96_tiprack_50uL_filter("tips_pcr")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10
tip_car[4] = tiprack_pcr

# Trough plate on rails=19, module slot 0
trough_module = hamilton_mfx_plateholder_DWP_metal_tapped("trough_module")
car_19 = hamilton_mfx_carrier_L5_base("car_19", modules={0:trough_module})
lh.deck.assign_child_resource(car_19, rails=19)
ab_trough = AGenBio_1_troughplate_100000uL_Fl("ab_trough")
trough_module.assign_child_resource(ab_trough)

# BioER DW plate rails=13, module slot 0,1
module_holding_dw_plate = hamilton_mfx_plateholder_DWP_metal_tapped("module_holding_dw_plate")
another_dw_plate_mod = hamilton_mfx_plateholder_DWP_metal_tapped("another_dw_plate_mod")
car_13 = hamilton_mfx_carrier_L5_base("car_13", modules={0:module_holding_dw_plate,1:another_dw_plate_mod})
lh.deck.assign_child_resource(car_13, rails=13)
dw_plate_sample = BioER_96_wellplate_Vb_2200uL("dw_plate_sample")
dw_plate_dilution = BioER_96_wellplate_Vb_2200uL("dw_plate_dilution")
module_holding_dw_plate.assign_child_resource(dw_plate_sample)
another_dw_plate_mod.assign_child_resource(dw_plate_dilution)

#
# PCR plates on rail 1 all modules, mastermix plate on module 0

pcr_mastermix_module = hamilton_mfx_plateholder_DWP_metal_tapped("pcr_mastermix_module")
pcr_sample_module_1 = hamilton_mfx_plateholder_DWP_metal_tapped("pcr_sample_module_1")
pcr_sample_module_2 = hamilton_mfx_plateholder_DWP_metal_tapped("pcr_sample_module_2")
pcr_sample_module_3 = hamilton_mfx_plateholder_DWP_metal_tapped("pcr_sample_module_3")
pcr_sample_module_4 = hamilton_mfx_plateholder_DWP_metal_tapped("pcr_sample_module_4")
pcr_mastermix_plate = VWR_96_wellplate_100_Vb_on_starCarrier_182070("pcr_mastermix_plate")
pcr_plate_1 = VWR_96_wellplate_100_Vb_on_starCarrier_182070("pcr_plate_1")
pcr_plate_2 = VWR_96_wellplate_100_Vb_on_starCarrier_182070("pcr_plate_2")
pcr_plate_3 = VWR_96_wellplate_100_Vb_on_starCarrier_182070("pcr_plate_3")
pcr_plate_4 = VWR_96_wellplate_100_Vb_on_starCarrier_182070("pcr_plate_4")
pcr_mastermix_module.assign_child_resource(pcr_mastermix_plate)
pcr_sample_module_1.assign_child_resource(pcr_plate_1)
pcr_sample_module_2.assign_child_resource(pcr_plate_2)
pcr_sample_module_3.assign_child_resource(pcr_plate_3)
pcr_sample_module_4.assign_child_resource(pcr_plate_4)
#car_1 = PLT_CAR_L5MD_A00("car_1", modules={0:pcr_mastermix_module, 1:pcr_sample_module_1, 2:pcr_sample_module_2, 3:pcr_sample_module_3, 4:pcr_sample_module_4})
car_1 = PLT_CAR_L5MD_A00("car_1",)
car_1[0].assign_child_resource(pcr_mastermix_plate)
car_1[1].assign_child_resource(pcr_plate_1)
car_1[2].assign_child_resource(pcr_plate_2)
car_1[3].assign_child_resource(pcr_plate_3)
car_1[4].assign_child_resource(pcr_plate_4)
lh.deck.assign_child_resource(car_1, rails=1)

# PCR specific resources 
# Rails =7 Deepwell plate module 1, water trough module 2

pcr_mix_module =hamilton_mfx_plateholder_DWP_metal_tapped("pcr_mix_module")
water_trough_module = hamilton_mfx_plateholder_DWP_metal_tapped("water_trough_module")
pcr_mix_plate = BioER_96_wellplate_Vb_2200uL("pcr_mix_plate") #~1.1 ml mix in each well of the first column
water_trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
pcr_mix_module.assign_child_resource(pcr_mix_plate)
water_trough_module.assign_child_resource(water_trough)
car_7 = hamilton_mfx_carrier_L5_base("car_7", modules={1:pcr_mix_module,2:water_trough_module})
lh.deck.assign_child_resource(car_7, rails=7)

CHANNELS_8=[0,1,2,3,4,5,6,7]



In [6]:
lh.summary()

Rail  Resource                       Type                 Coordinates (mm)
(-6)  ├── trash_core96               Trash                (-58.200, 106.000, 216.400)
      │
(1)   ├── car_1                      PlateCarrier         (100.000, 063.000, 100.000)
      │   ├── pcr_mastermix_plate    Plate                (104.000, 071.500, 209.600)
      │   ├── pcr_plate_1            Plate                (104.000, 167.500, 209.600)
      │   ├── pcr_plate_2            Plate                (104.000, 263.500, 209.600)
      │   ├── pcr_plate_3            Plate                (104.000, 359.500, 209.600)
      │   ├── pcr_plate_4            Plate                (104.000, 455.500, 209.600)
      │
(7)   ├── car_7                      MFXCarrier           (235.000, 063.000, 100.000)
      │   ├── pcr_mix_plate          Plate                (239.000, 168.000, 179.210)
      │   ├── water_trough           Plate                (239.000, 264.000, 179.210)
      │
(13)  ├── car_13                     MFXC

In [7]:
async def prep_master_mix_plate():
    pvol = 1000
    await lh.pick_up_tips(tiprack_1000["A1:H1"], use_channels=CHANNELS_8)
    well_A1 = pcr_mix_plate.get_item("A1")
    lheight = well_A1.compute_height_from_volume(pvol)
    await lh.aspirate(
        pcr_mix_plate["A1:H1"],
        vols=[500]*8,
        use_channels=CHANNELS_8,
        liquid_height=[lheight-12]*8,
        # lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        # immersion_depth=[8]*8,
        # suface_following_distance=[4]*8,
        # auto_surface_following=True,
        #mix=[Mix(volume=300, repetitions=3, flow_rate=100)]*8,
        flow_rate=100,
        settling_time=[1]*8,
        transport_air_volume=[0]*8
    )
    for col in range(1,7): #Not inclusive, go one more then you think you need
        await lh.dispense(
            pcr_mastermix_plate[f"A{col}:H{col}"],
            vols=[72]*8,
            use_channels=CHANNELS_8,
            flow_rate=72,
            transport_air_volume=[0]*8
        )

    await lh.dispense(
        pcr_mix_plate["A1:H1"],
        # vols=[30]*8,
        vols=[18]*8, # 450-72*6 = 450-432 = 18
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        suface_following_distance=[1]*8,
        blow_out=[1]*8,
        settling_time=[1]*8
        
    )

    await lh.aspirate(
        pcr_mix_plate["A1:H1"],
        vols=[500]*8,
        use_channels=CHANNELS_8,
        liquid_height=[1]*8,
        flow_rate=72,
        transport_air_volume=[0]*8
        # lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        # suface_following_distance=[1]*8,
        # mix=[Mix(volume=300, repetitions=3, flow_rate=100)]*8
     )

    for col in range(7,13): #Not inclusive, go one more then you think you need
         await lh.dispense(
            pcr_mastermix_plate[f"A{col}:H{col}"],
            vols=[72]*8,
            use_channels=CHANNELS_8,
            transport_air_volume=[0]*8
         )

    await lh.dispense(
        pcr_mix_plate["A1:H1"],
        vols=[2]*8,
        use_channels=CHANNELS_8,
        liquid_height=[1]*8,
        # lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        # suface_following_distance=[1]*8,
        blow_out=[True]*8,
        settling_time=[1]*8
        
    )
    #await lh.drop_tips(tiprack_1000["A1:H1"], use_channels=CHANNELS_8) #For testing only
    await lh.discard_tips()

In [8]:
def use_tips(col): #This funtion MUST be given an int or it will break
    row =(col // 2 + (col % 2 > 0))+1 
    if col % 2 == 0:
        return tiprack_1000[f"A{row}:D{row}"]
    else:
        return tiprack_1000[f"E{row}:H{row}"]
    #This function makes the half channel removal take entier coulmns insted of just the top half, should also mean the tip rack does not need to be refilled during a run


In [ ]:
half_channels=[4,5,6,7]
async def mix_dilute_one_column(column):
    await lh.pick_up_tips(use_tips(column), use_channels=half_channels)
    await lh.aspirate(
        dw_plate_sample[f"A{column}", f"C{column}", f"E{column}", f"G{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[1]*4,
        liquid_height=[1]*4,
        mix=[Mix(volume=500, repetitions=3, flow_rate=250)]*4
    )

    await lh.dispense(
        dw_plate_sample[f"B{column}", f"D{column}", f"F{column}", f"H{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        liquid_height=[1]*4,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[2]*4,
        mix=[Mix(volume=500, repetitions=3, flow_rate=250)]*4,
        blow_out=[True]*4
    )

    await lh.aspirate(
        dw_plate_sample[f"B{column}", f"D{column}", f"F{column}", f"H{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        liquid_height=[1]*4,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[1]*4
    )

    await lh.dispense(
        dw_plate_dilution[f"A{column}", f"C{column}", f"E{column}", f"G{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[2]*4,
        liquid_height=[1]*4,
        mix=[Mix(volume=500, repetitions=3, flow_rate=250)]*4,
        blow_out=[True]*4
    )

    await lh.aspirate(
        dw_plate_dilution[f"B{column}", f"D{column}", f"F{column}", f"H{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        liquid_height=[1]*4,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[1]*4
    )

    await lh.dispense(
        dw_plate_dilution[f"B{column}", f"D{column}", f"F{column}", f"H{column}"],
        vols=[100]*4,
        use_channels=half_channels,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*4,
        #suface_following_distance=[2]*4,
        liquid_height=[1]*4,
        mix=[Mix(volume=500, repetitions=3, flow_rate=250)]*4,
        blow_out=[True]*4
    )

    #await lh.drop_tips(use_tips(column), use_channels=half_channels) #For testing only
    await lh.discard_tips()


In [10]:
async def make_one_column_pcr(column): #Very inconsitnte volumes use make_one_column_pcr_alt instead
    await lh.pick_up_tips(tiprack_pcr[f"A{column}:H{column}"], use_channels=CHANNELS_8)

    await lh.aspirate(
        dw_plate_sample[f"A{column}:H{column}"],
        vols=[18]*8,
        use_channels=CHANNELS_8,
        liquid_height=[1]*8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[3]*8,
        #mix=[Mix(volume=40, repetitions=3, flow_rate=40)]*8
    )

    await lh.dispense(
        pcr_mastermix_plate[f"A{column}:H{column}"],
        vols=[18]*8,
        use_channels=CHANNELS_8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[3]*8,
        liquid_height=[0]*8,
        mix=[Mix(volume=45, repetitions=2, flow_rate=20)]*8,
        settling_time=[1]*8,
        blow_out=[True]*8
    )

    await lh.aspirate(
        pcr_mastermix_plate[f"A{column}:H{column}"],
        vols=[45]*8,
        use_channels=CHANNELS_8,
        mix=[Mix(volume=45, repetitions=2, flow_rate=20)]*8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[3]*8
        liquid_height=[1]*8,
    )

    await lh.dispense(
        pcr_plate_1[f"A{column}:H{column}"],
        vols=[20]*8,
        use_channels=CHANNELS_8,
        liquid_height=[0]*8,
        flow_rate=[10]*8
    )

    await lh.dispense(
        pcr_plate_2[f"A{column}:H{column}"],
        vols=[20]*8,
        use_channels=CHANNELS_8,
        liquid_height=[0]*8,
        flow_rate=[10]*8

    )

    await lh.dispense(
        pcr_mastermix_plate[f"A{column}:H{column}"],
        vols=[2]*8,
        use_channels=CHANNELS_8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[1]*8,
        liquid_height=[1]*8,
        #mix=[Mix(volume=40, repetitions=3, flow_rate=40)]*8,
        blow_out=[True]*8
    )

    await lh.aspirate(
        pcr_mastermix_plate[f"A{column}:H{column}"],
        vols=[45]*8,
        use_channels=CHANNELS_8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[2]*8
        liquid_height=[0]*8,

    )

    await lh.dispense(
        pcr_plate_3[f"A{column}:H{column}"],
        vols=[20]*8,
        use_channels=CHANNELS_8,
        liquid_height=[0.5]*8,
        flow_rate=[10]*8
    )

    await lh.dispense(
        pcr_plate_4[f"A{column}:H{column}"],
        vols=[20]*8,
        use_channels=CHANNELS_8,
        liquid_height=[0.5]*8,
        flow_rate=[10]*8
    )

    await lh.discard_tips()

In [11]:
async def make_one_column_pcr_alt(column):
    await lh.pick_up_tips(tiprack_pcr[f"A{column}:H{column}"], use_channels=CHANNELS_8)

    await lh.aspirate(
        dw_plate_sample[f"A{column}:H{column}"],
        vols=[18]*8,
        use_channels=CHANNELS_8,
        liquid_height=[1]*8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[3]*8,
        #mix=[Mix(volume=40, repetitions=3, flow_rate=40)]*8
    )

    await lh.dispense(
        pcr_mastermix_plate[f"A{column}:H{column}"],
        vols=[18]*8,
        use_channels=CHANNELS_8,
        #lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        #suface_following_distance=[3]*8,
        liquid_height=[0]*8,
        mix=[Mix(volume=45, repetitions=4, flow_rate=20)]*8,
        settling_time=[1]*8,
        blow_out=[True]*8
    )

    for plate in range(1,5):
        await lh.aspirate(
            pcr_mastermix_plate[f"A{column}:H{column}"],
            vols=[25]*8,
            use_channels=CHANNELS_8,
            liquid_height=[0.2]*8,
            transport_air_volume=[0]*8
        )

        await lh.dispense(
            eval(f"pcr_plate_{plate}[f'A{column}:H{column}']"),
            vols=[20]*8,
            use_channels=CHANNELS_8,
            liquid_height=[0]*8,
            blow_out=[True]*8
        )

        await lh.dispense(
            eval(f"pcr_plate_{plate}[f'A{column}:H{column}']"),
            vols=[5]*8,
            use_channels=CHANNELS_8,
            liquid_height=[0]*8,
            blow_out=[True]*8
        )


    await lh.discard_tips()

In [ ]:
await prep_master_mix_plate()
#~3min

/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot/pylabrobot/liquid_handling/liquid_handler.py:345: UserWarning: Extra arguments to backend.aspirate: {'flow_rate'}
  warnings.warn(f"Extra arguments to backend.{method.__name__}: {extra}")
/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot/pylabrobot/liquid_handling/liquid_handler.py:345: UserWarning: Extra arguments to backend.dispense: {'flow_rate'}
  warnings.warn(f"Extra arguments to backend.{method.__name__}: {extra}")
/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot/pylabrobot/liquid_handling/liquid_handler.py:345: UserWarning: Extra arguments to backend.dispense: {'suface_following_distance'}
  warnings.warn(f"Extra arguments to backend.{method.__name__}: {extra}")


ChannelizedError: ChannelizedError(errors={0: HardwareError('Z-drive movement error'), 1: HardwareError('Z-drive movement error'), 2: HardwareError('Z-drive movement error'), 3: HardwareError('Z-drive movement error'), 4: HardwareError('Z-drive movement error'), 5: HardwareError('Z-drive movement error'), 6: HardwareError('Z-drive movement error'), 7: HardwareError('Z-drive movement error')}, raw_response=C0TRid0025er99/00 P102/62 P202/62 P302/62 P402/62 P502/62 P602/62 P702/62 P802/62kz370 382 390 351 436 349 369 407vz000 000 000 000 000 000 000 000)

In [ ]:

for col in range(1,13): #(1,13) for full plate
   await mix_dilute_one_column(col)
   await make_one_column_pcr_alt(col)




In [ ]:
#####
##Testing things
########

#await prep_master_mix_plate()
#await mix_dilute_one_column(12)
#await make_one_column_pcr_alt(11)
await lh.stop()

In [ ]:
##await lh.drop_tips(tiprack_1000["A12:D12"], use_channels=half_channels)
##await lh.drop_tips(tiprack_1000["A1:H1"], use_channels=CHANNELS_8)
##await lh.drop_tips(tiprack_pcr[f"A{11}:H{11}"], use_channels=CHANNELS_8)
await lh.discard_tips()
